# Master Dataset Creation

In this notebook, 15 tables are loaded containing socio-demographic, economic, and healthcare characteristics of Russian regions.  
Each table has already undergone an EDA step.

The following steps are performed in this notebook:
- loading the final cleaned tables;
- filtering data for the years (2015–2020);
- validating and aligning region names and data structure;
- partial aggregation where necessary;
- merging all tables into a single master dataset;
- saving the final master dataset for further analysis.

In [36]:
import pandas as pd

# Path to folder with final cleaned datasets
data_path = "../../data/clean/"

# Load datasets
df_poverty = pd.read_csv(data_path + "poverty_AfterEDA.csv")
df_income = pd.read_csv(data_path + "income_AfterEDA.csv")
df_socdem = pd.read_csv(data_path + "socdem_AfterEDA.csv")
df_drug_alco = pd.read_csv(data_path + "drug_alco_AfterEDA.csv")
df_morbidity = pd.read_csv(data_path + "morbidity_AfterEDA.csv")
df_newborns = pd.read_csv(data_path + "newborns_AfterEDA.csv")
df_housing = pd.read_csv(data_path + "housing_AfterEDA.csv")
df_gross = pd.read_csv(data_path + "gross_AfterEDA.csv")
df_population = pd.read_csv(data_path + "population_AfterEDA.csv")
df_disabled = pd.read_csv(data_path + "disabled_AfterEDA.csv")
df_child_mortality_rural = pd.read_csv(data_path + "child_rural_AfterEDA.csv")
df_child_mortality_urban = pd.read_csv(data_path + "child_urban_AfterEDA.csv")
df_retail = pd.read_csv(data_path + "retail_AfterEDA.csv")
df_welfare = pd.read_csv(data_path + "welfare_AfterEDA.csv")
df_regional_production = pd.read_csv(data_path + "reg_production_AfterEDA.csv")

In [37]:
# Find the year column and filter by selected range
def filter_years(df, years=None):
    # Possible names for the year column
    possible_cols = ["year", "год", "Год"]
    
    col = None
    for c in possible_cols:
        if c in df.columns:
            col = c
            break

    if col is None:
        raise ValueError("No 'year', 'год', or 'Год' column found in the dataset.")
    
    # Default range: 2015–2020
    if years is None:
        years = list(range(2015, 2021))
    
    return df[df[col].isin(years)]


# Apply filtering to datasets that contain year information
df_poverty = filter_years(df_poverty)
df_income = filter_years(df_income)
df_socdem = filter_years(df_socdem)
df_drug_alco = filter_years(df_drug_alco)
df_gross = filter_years(df_gross)
df_population = filter_years(df_population)
df_child_mortality_rural = filter_years(df_child_mortality_rural)
df_child_mortality_urban = filter_years(df_child_mortality_urban)
df_retail = filter_years(df_retail)
df_welfare = filter_years(df_welfare)
df_regional_production = filter_years(df_regional_production)
df_newborns = filter_years(df_newborns)
df_disabled = filter_years(df_disabled)

In [38]:
# Keep only 2015 and 2016, then aggregate cases
morbidity_agg = (
    df_morbidity
    [df_morbidity["year"].isin([2015, 2016])]
    .groupby(["region_standard", "year"])["cases"]
    .sum()
    .reset_index()
)

In [39]:
# Convert to wide format
drug_alco_pivot = (
    df_drug_alco
    .pivot_table(
        index=["region_standard", "year"],
        columns="diagnosis",
        values="value"
    )
    .reset_index()
)

# Rename columns
drug_alco_pivot.rename(columns={
    "alcohol": "alcohol_rate",
    "drugs": "drugs_rate"
}, inplace=True)

In [40]:
# Aggregate production volume by region and year
regional_production_agg = (
    df_regional_production
    .groupby(["region_standard", "year"])["value"]
    .sum()
    .reset_index()
    .rename(columns={"value": "production_total"})
)

In [41]:
# Dictionary of tables
tables = {
    "poverty": df_poverty,
    "income": df_income,
    "socdem": df_socdem,
    "drug_alco": drug_alco_pivot,
    "newborns": df_newborns,
    "disabled": df_disabled,
    "morbidity": morbidity_agg,
    "housing": df_housing,
    "workers": df_gross,
    "population": df_population,
    "child_mortality_rural": df_child_mortality_rural,
    "child_mortality_urban": df_child_mortality_urban,
    "retail": df_retail,
    "welfare": df_welfare,
    "regional_production": regional_production_agg
}

# Standardize key columns
for name, df in tables.items():
    df.rename(columns={
        "Регион": "region",
        "Год": "year",
        "год": "year",
        "Region": "region",
        "Year": "year"
    }, inplace=True)
    
    # Keep only region_standard
    if "region" in df.columns and "region_standard" not in df.columns:
        df.rename(columns={"region": "region_standard"}, inplace=True)
    elif "region" in df.columns and "region_standard" in df.columns:
        df.drop(columns=["region"], inplace=True)

In [42]:
# Remove duplicates based on region_standard + year
for name, df in tables.items():
    before = df.shape[0]
    df.drop_duplicates(subset=["region_standard", "year"], inplace=True)
    after = df.shape[0]
    print(f"{name:<25} — removed {before - after} duplicates")

poverty                   — removed 12 duplicates
income                    — removed 0 duplicates
socdem                    — removed 8 duplicates
drug_alco                 — removed 0 duplicates
newborns                  — removed 12 duplicates
disabled                  — removed 0 duplicates
morbidity                 — removed 0 duplicates
housing                   — removed 0 duplicates
workers                   — removed 12 duplicates
population                — removed 7 duplicates
child_mortality_rural     — removed 12 duplicates
child_mortality_urban     — removed 12 duplicates
retail                    — removed 14 duplicates
welfare                   — removed 0 duplicates
regional_production       — removed 0 duplicates


In [43]:
# Start building the master dataset from the poverty table
master_df = df_poverty.copy()

# Income
master_df = master_df.merge(df_income, on=["region_standard", "year"], how="left")

# Socio-demographics
master_df = master_df.merge(df_socdem, on=["region_standard", "year"], how="left")

# Alcohol and drugs
master_df = master_df.merge(drug_alco_pivot, on=["region_standard", "year"], how="left")

# Morbidity
master_df = master_df.merge(morbidity_agg, on=["region_standard", "year"], how="left")

# Births
master_df = master_df.merge(df_newborns, on=["region_standard", "year"], how="left")

# Disability
master_df = master_df.merge(df_disabled, on=["region_standard", "year"], how="left")

# GRP per capita
master_df = master_df.merge(df_gross, on=["region_standard", "year"], how="left")

# Population
master_df = master_df.merge(df_population, on=["region_standard", "year"], how="left")

# Rural child mortality
master_df = master_df.merge(df_child_mortality_rural, on=["region_standard", "year"], how="left")

# Urban child mortality
master_df = master_df.merge(df_child_mortality_urban, on=["region_standard", "year"], how="left")

# Retail
master_df = master_df.merge(df_retail, on=["region_standard", "year"], how="left")

# Welfare
master_df = master_df.merge(df_welfare, on=["region_standard", "year"], how="left")

# Regional production
master_df = master_df.merge(regional_production_agg, on=["region_standard", "year"], how="left")

In [44]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    510 non-null    int64  
 1   poverty_percent         510 non-null    float64
 2   region_standard         510 non-null    object 
 3   income_per_capita       478 non-null    float64
 4   real_income             478 non-null    float64
 5   nominal_wage            478 non-null    float64
 6   real_wage               478 non-null    float64
 7   children_under_16       328 non-null    float64
 8   above_working_age       328 non-null    float64
 9   working_age_population  328 non-null    float64
 10  alcohol_rate            306 non-null    float64
 11  drugs_rate              306 non-null    float64
 12  cases                   170 non-null    float64
 13  births                  510 non-null    float64
 14  total                   340 non-null    fl

In [45]:
# Rename columns
master_df.rename(columns={
    "region_standard": "region",
    "children_under_16": "children_percent",
    "above_working_age": "elderly_percent",
    "working_age_population": "working_age_percent",
    "cases": "crime_cases",
    "ВРП_на_душу": "gdp_per_capita",
    "infant_mortality_x": "infant_mortality_rural",
    "infant_mortality_y": "infant_mortality_urban"
}, inplace=True)

In [46]:
# Only numerical features
df_numeric = master_df.select_dtypes(include='number')

# Compute missing value ratio
missing_ratio = df_numeric.isnull().mean().sort_values(ascending=False)

# Show features with missing values
missing_ratio[missing_ratio > 0]


crime_cases            0.666667
drugs_rate             0.400000
alcohol_rate           0.400000
children_percent       0.356863
elderly_percent        0.356863
working_age_percent    0.356863
60_                    0.333333
51_60                  0.333333
41_50                  0.333333
31_40                  0.333333
18_30                  0.333333
total                  0.333333
real_wage              0.062745
nominal_wage           0.062745
real_income            0.062745
income_per_capita      0.062745
welfare_percent        0.058824
population             0.023529
dtype: float64

In [47]:
# Fill missing values with regional median
columns_to_fill = [
    "children_percent", "elderly_percent", "working_age_percent",
    "alcohol_rate", "drugs_rate",
    "total", "18_30", "31_40",
    "41_50", "51_60", "60_",
    "nominal_wage", "real_wage", "income_per_capita", "real_income",
    "welfare_percent", "population"
]

for col in columns_to_fill:
    master_df[col] = master_df.groupby("region")[col].transform(lambda x: x.fillna(x.median()))

# Drop heavily corrupted feature
master_df.drop(columns=["cases", "crime_cases"], inplace=True, errors="ignore")

# Final fill for remaining missing values using global median
master_df.fillna(master_df.median(numeric_only=True), inplace=True)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  

In [48]:
# --- Add derived features for further analysis ---

# Dependency ratio: children + elderly
master_df["dependent_percent"] = (master_df["children_percent"] + master_df["elderly_percent"])

# Number of disabled people per 1,000 population
master_df["disabled_rate_per_1000"] = (master_df["total"] / master_df["population"]) * 1000

# Combined rate of alcoholism and drug addiction
master_df["addiction_rate"] = (master_df["alcohol_rate"] + master_df["drugs_rate"])

# Production per capita
master_df["production_per_capita"] = (master_df["production_total"] / master_df["population"])

# Ratio of poverty to social welfare spending
master_df["poverty_welfare_ratio"] = (master_df["poverty_percent"] / master_df["welfare_percent"])

# Birth rate per 1,000 population
master_df["birth_rate_per_1000"] = master_df["births"] / master_df["population"] * 1000

# Rural infant mortality per 1,000 births
master_df["infant_mortality_rural_rate"] = master_df["infant_mortality_rural"] / master_df["births"] * 1000

# Urban infant mortality per 1,000 births
master_df["infant_mortality_urban_rate"] = master_df["infant_mortality_urban"] / master_df["births"] * 1000

# Disabled people aged 18–30 per 1,000 population
master_df["disabled_18_30_rate"] = master_df["18_30"] / master_df["population"] * 1000

# Disabled people aged 31–40 per 1,000 population
master_df["disabled_31_40_rate"] = master_df["31_40"] / master_df["population"] * 1000

# Disabled people aged 41–50 per 1,000 population
master_df["disabled_41_50_rate"] = master_df["41_50"] / master_df["population"] * 1000

# Disabled people aged 51–60 per 1,000 population
master_df["disabled_51_60_rate"] = master_df["51_60"] / master_df["population"] * 1000

# Disabled people aged 60+ per 1,000 population
master_df["disabled_60_plus_rate"] = master_df["60_"] / master_df["population"] * 1000

In [49]:
master_df.to_csv("../../data/clean/master_dataset_2015_2020.csv", index=False)